# Actividad integradora: criptosistema asimétrico educativo

Este notebook implementa un esquema RSA educativo para firma digital y verificación entre dos personas.

La simulación usa dos participantes:

- Persona 1
- Persona 2

Cada persona genera su propio par de llaves. Cada una firma su propio mensaje usando su llave privada. Después, la otra persona verifica el mensaje usando la llave pública correspondiente.

El notebook también guarda evidencias en archivos `.txt` separados dentro de la carpeta `evidencias`.

## 1. Importación de módulos permitidos

Se usan módulos generales de Python.

No se usan librerías criptográficas que generen llaves, firmen o verifiquen automáticamente. `hashlib` se usa únicamente para calcular SHA-256 del mensaje.

In [1]:
from pathlib import Path
import hashlib
import math
import random

## 2. Carpeta de evidencias

Esta sección crea una carpeta llamada `evidencias`.

Dentro de ella se separan los archivos de Persona 1, Persona 2, intercambio y pruebas inválidas. Estos archivos sirven para demostrar el funcionamiento del sistema paso por paso.

In [2]:
CARPETA_EVIDENCIAS = Path("evidencias")

CARPETA_PERSONA_1 = CARPETA_EVIDENCIAS / "persona_1"
CARPETA_PERSONA_2 = CARPETA_EVIDENCIAS / "persona_2"
CARPETA_INTERCAMBIO = CARPETA_EVIDENCIAS / "intercambio"
CARPETA_PRUEBAS_INVALIDAS = CARPETA_EVIDENCIAS / "pruebas_invalidas"

for carpeta in [
    CARPETA_PERSONA_1,
    CARPETA_PERSONA_2,
    CARPETA_INTERCAMBIO,
    CARPETA_PRUEBAS_INVALIDAS
]:
    carpeta.mkdir(parents=True, exist_ok=True)


def guardar_txt(ruta, contenido):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(str(contenido), encoding="utf-8")
    print(f"Archivo generado: {ruta}")

## 3. Funciones matemáticas auxiliares

RSA necesita operaciones matemáticas básicas.

El algoritmo extendido de Euclides permite calcular el inverso modular. Ese inverso modular se usa para obtener el exponente privado `d`, que forma parte de la llave privada.

In [3]:
def algoritmo_extendido_euclides(a, b):
    if b == 0:
        return a, 1, 0

    mcd, x1, y1 = algoritmo_extendido_euclides(b, a % b)
    x = y1
    y = x1 - (a // b) * y1

    return mcd, x, y


def inverso_modular(a, modulo):
    mcd, x, _ = algoritmo_extendido_euclides(a, modulo)

    if mcd != 1:
        raise ValueError("No existe inverso modular porque los valores no son coprimos.")

    return x % modulo

## 4. Generación de primos

Para generar llaves RSA se necesitan dos números primos secretos.

Esta implementación usa una prueba probabilística de primalidad tipo Miller-Rabin. Es suficiente para una implementación educativa, aunque no representa una implementación industrial completa.

In [4]:
def es_probablemente_primo(numero, rondas=12):
    if numero < 2:
        return False

    primos_pequenos = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31]

    if numero in primos_pequenos:
        return True

    for primo in primos_pequenos:
        if numero % primo == 0:
            return False

    d = numero - 1
    s = 0

    while d % 2 == 0:
        d //= 2
        s += 1

    for _ in range(rondas):
        a = random.randrange(2, numero - 2)
        x = pow(a, d, numero)

        if x == 1 or x == numero - 1:
            continue

        for _ in range(s - 1):
            x = pow(x, 2, numero)

            if x == numero - 1:
                break
        else:
            return False

    return True


def generar_primo(bits=256):
    while True:
        candidato = random.getrandbits(bits)
        candidato |= (1 << bits - 1)
        candidato |= 1

        if es_probablemente_primo(candidato):
            return candidato

## 5. Generación de llaves

Cada persona genera su propio par de llaves.

La llave pública está formada por `(e, n)` y puede compartirse.

La llave privada está formada por `(d, n)` y debe mantenerse en secreto.

Los valores `p`, `q` y `phi` solo se guardan como evidencia educativa para explicar cómo se generaron las llaves.

In [5]:
def generar_llaves(bits=256):
    p = generar_primo(bits)
    q = generar_primo(bits)

    while p == q:
        q = generar_primo(bits)

    n = p * q
    phi = (p - 1) * (q - 1)

    e = 65537

    if math.gcd(e, phi) != 1:
        e = 3

        while math.gcd(e, phi) != 1:
            e += 2

    d = inverso_modular(e, phi)

    llave_publica = {
        "e": e,
        "n": n
    }

    llave_privada = {
        "d": d,
        "n": n
    }

    detalles = {
        "p": p,
        "q": q,
        "phi": phi
    }

    return llave_publica, llave_privada, detalles

## 6. Preparación del mensaje

RSA opera con números, no directamente con texto.

Por eso el mensaje se convierte a bytes usando UTF-8, luego se calcula su hash SHA-256 y finalmente se convierte ese hash a número entero. El valor se reduce módulo `n` para poder usarlo dentro del esquema RSA.

Si el mensaje cambia, el hash cambia y la firma deja de ser válida.

In [6]:
def preparar_mensaje(mensaje, n):
    if not isinstance(mensaje, str):
        raise TypeError("El mensaje debe ser una cadena de texto.")

    mensaje_bytes = mensaje.encode("utf-8")
    hash_hexadecimal = hashlib.sha256(mensaje_bytes).hexdigest()
    hash_entero = int(hash_hexadecimal, 16)
    valor_preparado = hash_entero % n

    return {
        "mensaje_original": mensaje,
        "hash_hexadecimal": hash_hexadecimal,
        "hash_entero": hash_entero,
        "valor_preparado": valor_preparado
    }

## 7. Firma digital

Cada persona firma su propio mensaje con su llave privada.

La firma no es el mensaje original. La firma es un número generado a partir del hash preparado del mensaje y del exponente privado `d`.

In [7]:
def firmar_mensaje(mensaje, llave_privada):
    if not isinstance(llave_privada, dict):
        raise TypeError("La llave privada debe tener formato de diccionario.")

    if "d" not in llave_privada or "n" not in llave_privada:
        raise ValueError("La llave privada debe contener los valores d y n.")

    d = llave_privada["d"]
    n = llave_privada["n"]

    preparado = preparar_mensaje(mensaje, n)
    firma = pow(preparado["valor_preparado"], d, n)

    return firma, preparado

## 8. Verificación de firma

Para verificar una firma se usa la llave pública de la persona que firmó el mensaje.

El proceso recupera un valor desde la firma usando la llave pública y lo compara contra el hash preparado del mensaje recibido.

Si coinciden, la firma es válida. Si no coinciden, la firma es inválida.

In [8]:
def verificar_firma(mensaje, firma, llave_publica):
    try:
        if not isinstance(llave_publica, dict):
            return False, "La llave pública no tiene un formato válido.", None

        if "e" not in llave_publica or "n" not in llave_publica:
            return False, "La llave pública debe contener los valores e y n.", None

        if not isinstance(firma, int):
            return False, "La firma debe ser un número entero.", None

        e = llave_publica["e"]
        n = llave_publica["n"]

        if firma < 0 or firma >= n:
            return False, "La firma está fuera del rango válido.", None

        preparado = preparar_mensaje(mensaje, n)
        valor_recuperado = pow(firma, e, n)

        datos = {
            "hash_mensaje_recibido": preparado["hash_hexadecimal"],
            "valor_preparado_mensaje_recibido": preparado["valor_preparado"],
            "valor_recuperado_desde_firma": valor_recuperado
        }

        if valor_recuperado == preparado["valor_preparado"]:
            return True, "La firma es válida.", datos

        return False, "La firma no corresponde al mensaje o a la llave pública usada.", datos

    except Exception as error:
        return False, f"Error durante la verificación: {error}", None

## 9. Formatos para guardar evidencias

Estas funciones convierten los datos importantes en texto legible.

Así, cada llave, mensaje, hash, firma y verificación se puede guardar en un archivo `.txt` diferente.

In [9]:
def formato_llave_publica(nombre, llave):
    return f'''Llave pública de {nombre}

Esta llave puede compartirse.
Sirve para verificar firmas creadas con la llave privada correspondiente.

e = {llave["e"]}

n = {llave["n"]}
'''


def formato_llave_privada(nombre, llave):
    return f'''Llave privada de {nombre}

Esta llave debe mantenerse en secreto.
Sirve para firmar mensajes.

d = {llave["d"]}

n = {llave["n"]}
'''


def formato_detalles(nombre, detalles):
    return f'''Detalles educativos de generación de llaves de {nombre}

Estos valores se muestran solo para explicar cómo se generaron las llaves.
En un sistema real no deberían compartirse.

p = {detalles["p"]}

q = {detalles["q"]}

phi = {detalles["phi"]}
'''


def formato_hash(nombre, preparado):
    return f'''Preparación del mensaje de {nombre}

Mensaje original:
{preparado["mensaje_original"]}

Hash SHA-256:
{preparado["hash_hexadecimal"]}

Hash convertido a entero:
{preparado["hash_entero"]}

Valor preparado dentro del módulo n:
{preparado["valor_preparado"]}
'''


def formato_firma(nombre, firma):
    return f'''Firma digital de {nombre}

La firma fue generada usando la llave privada de {nombre}.
Esta firma se comparte junto con el mensaje y la llave pública para que otra persona pueda verificarla.

firma = {firma}
'''


def formato_verificacion(titulo, mensaje, firma, llave_publica, resultado, explicacion, datos):
    texto = f'''{titulo}

Mensaje recibido:
{mensaje}

Firma recibida:
{firma}

Llave pública usada:
e = {llave_publica.get("e", "no disponible")}
n = {llave_publica.get("n", "no disponible")}

Resultado:
{resultado}

Explicación:
{explicacion}
'''

    if datos:
        texto += f'''
Hash del mensaje recibido:
{datos["hash_mensaje_recibido"]}

Valor preparado del mensaje recibido:
{datos["valor_preparado_mensaje_recibido"]}

Valor recuperado desde la firma usando la llave pública:
{datos["valor_recuperado_desde_firma"]}

Comparación:
La firma es válida solo si el valor recuperado desde la firma coincide con el valor preparado del mensaje recibido.
'''

    return texto

## 10. Generación de llaves para Persona 1 y Persona 2

Aquí se generan dos pares de llaves independientes.

Persona 1 tendrá su propia llave pública y privada. Persona 2 también tendrá su propia llave pública y privada.

In [10]:
random.seed(132)

llave_publica_1, llave_privada_1, detalles_1 = generar_llaves(bits=256)
llave_publica_2, llave_privada_2, detalles_2 = generar_llaves(bits=256)

guardar_txt(CARPETA_PERSONA_1 / "01_llave_publica_persona_1.txt", formato_llave_publica("Persona 1", llave_publica_1))
guardar_txt(CARPETA_PERSONA_1 / "02_llave_privada_persona_1.txt", formato_llave_privada("Persona 1", llave_privada_1))
guardar_txt(CARPETA_PERSONA_1 / "03_detalles_generacion_persona_1.txt", formato_detalles("Persona 1", detalles_1))

guardar_txt(CARPETA_PERSONA_2 / "01_llave_publica_persona_2.txt", formato_llave_publica("Persona 2", llave_publica_2))
guardar_txt(CARPETA_PERSONA_2 / "02_llave_privada_persona_2.txt", formato_llave_privada("Persona 2", llave_privada_2))
guardar_txt(CARPETA_PERSONA_2 / "03_detalles_generacion_persona_2.txt", formato_detalles("Persona 2", detalles_2))

print("Llaves generadas correctamente para Persona 1 y Persona 2.")

Archivo generado: evidencias\persona_1\01_llave_publica_persona_1.txt
Archivo generado: evidencias\persona_1\02_llave_privada_persona_1.txt
Archivo generado: evidencias\persona_1\03_detalles_generacion_persona_1.txt
Archivo generado: evidencias\persona_2\01_llave_publica_persona_2.txt
Archivo generado: evidencias\persona_2\02_llave_privada_persona_2.txt
Archivo generado: evidencias\persona_2\03_detalles_generacion_persona_2.txt
Llaves generadas correctamente para Persona 1 y Persona 2.


## 11. Mensajes de Persona 1 y Persona 2

Cada persona tiene un mensaje diferente.

Estos mensajes serán firmados con la llave privada de quien los escribió.

In [11]:
mensaje_1 = "Que onda David ¿como estas?, ¿listo para la actividad?."
mensaje_2 = "Hola muy bien aqui chambeando ya tengo todo listo para pensar ¿y tu?."

guardar_txt(CARPETA_PERSONA_1 / "04_mensaje_persona_1.txt", f"Mensaje de Persona 1:\n\n{mensaje_1}\n")
guardar_txt(CARPETA_PERSONA_2 / "04_mensaje_persona_2.txt", f"Mensaje de Persona 2:\n\n{mensaje_2}\n")

print("Mensaje de Persona 1:")
print(mensaje_1)
print()
print("Mensaje de Persona 2:")
print(mensaje_2)

Archivo generado: evidencias\persona_1\04_mensaje_persona_1.txt
Archivo generado: evidencias\persona_2\04_mensaje_persona_2.txt
Mensaje de Persona 1:
Que onda David ¿como estas?, ¿listo para la actividad?.

Mensaje de Persona 2:
Hola muy bien aqui chambeando ya tengo todo listo para pensar ¿y tu?.


## 12. Firma de los mensajes

Persona 1 firma su mensaje con la llave privada de Persona 1.

Persona 2 firma su mensaje con la llave privada de Persona 2.

Después se guardan el hash y la firma de cada persona.

In [12]:
firma_1, preparado_1 = firmar_mensaje(mensaje_1, llave_privada_1)
firma_2, preparado_2 = firmar_mensaje(mensaje_2, llave_privada_2)

guardar_txt(CARPETA_PERSONA_1 / "05_hash_mensaje_persona_1.txt", formato_hash("Persona 1", preparado_1))
guardar_txt(CARPETA_PERSONA_1 / "06_firma_persona_1.txt", formato_firma("Persona 1", firma_1))

guardar_txt(CARPETA_PERSONA_2 / "05_hash_mensaje_persona_2.txt", formato_hash("Persona 2", preparado_2))
guardar_txt(CARPETA_PERSONA_2 / "06_firma_persona_2.txt", formato_firma("Persona 2", firma_2))

print("Firma de Persona 1:")
print(firma_1)
print()
print("Firma de Persona 2:")
print(firma_2)

Archivo generado: evidencias\persona_1\05_hash_mensaje_persona_1.txt
Archivo generado: evidencias\persona_1\06_firma_persona_1.txt
Archivo generado: evidencias\persona_2\05_hash_mensaje_persona_2.txt
Archivo generado: evidencias\persona_2\06_firma_persona_2.txt
Firma de Persona 1:
3748511031696063633874675177278308520035630115039663114990947306670888600368033396885689250548322729767815654245242163085222526452956585404199506170979501

Firma de Persona 2:
1812148069930065045086299304319987068896682632617334551518646517453997207887868081484970180108355920106128131058519979853412094993294191521323578448965346


## 13. Intercambio y verificación correcta

Persona 1 verifica el mensaje de Persona 2 usando la llave pública de Persona 2.

Persona 2 verifica el mensaje de Persona 1 usando la llave pública de Persona 1.

Este es el caso correcto, porque cada firma se verifica con la llave pública correspondiente.

In [13]:
resultado_p1_verifica_p2, explicacion_p1_verifica_p2, datos_p1_verifica_p2 = verificar_firma(
    mensaje_2,
    firma_2,
    llave_publica_2
)

resultado_p2_verifica_p1, explicacion_p2_verifica_p1, datos_p2_verifica_p1 = verificar_firma(
    mensaje_1,
    firma_1,
    llave_publica_1
)

guardar_txt(
    CARPETA_INTERCAMBIO / "01_persona_1_verifica_a_persona_2.txt",
    formato_verificacion(
        "Persona 1 verifica el mensaje de Persona 2",
        mensaje_2,
        firma_2,
        llave_publica_2,
        resultado_p1_verifica_p2,
        explicacion_p1_verifica_p2,
        datos_p1_verifica_p2
    )
)

guardar_txt(
    CARPETA_INTERCAMBIO / "02_persona_2_verifica_a_persona_1.txt",
    formato_verificacion(
        "Persona 2 verifica el mensaje de Persona 1",
        mensaje_1,
        firma_1,
        llave_publica_1,
        resultado_p2_verifica_p1,
        explicacion_p2_verifica_p1,
        datos_p2_verifica_p1
    )
)

print("Persona 1 verifica a Persona 2:")
print(resultado_p1_verifica_p2, explicacion_p1_verifica_p2)
print()
print("Persona 2 verifica a Persona 1:")
print(resultado_p2_verifica_p1, explicacion_p2_verifica_p1)

Archivo generado: evidencias\intercambio\01_persona_1_verifica_a_persona_2.txt
Archivo generado: evidencias\intercambio\02_persona_2_verifica_a_persona_1.txt
Persona 1 verifica a Persona 2:
True La firma es válida.

Persona 2 verifica a Persona 1:
True La firma es válida.


## 14. Valores recuperados desde las firmas

En una firma digital no se recupera el mensaje original.

Lo que se recupera usando la llave pública es un valor numérico derivado de la firma. Ese valor se compara contra el hash preparado del mensaje recibido.

Si ambos valores coinciden, la firma es válida.

In [14]:
guardar_txt(
    CARPETA_INTERCAMBIO / "03_valor_recuperado_firma_persona_1.txt",
    f'''Valor recuperado desde la firma de Persona 1

Al verificar la firma de Persona 1 con su llave pública, se obtiene este valor:

{datos_p2_verifica_p1["valor_recuperado_desde_firma"]}

Este valor debe compararse contra el valor preparado del mensaje recibido:

{datos_p2_verifica_p1["valor_preparado_mensaje_recibido"]}
'''
)

guardar_txt(
    CARPETA_INTERCAMBIO / "04_valor_recuperado_firma_persona_2.txt",
    f'''Valor recuperado desde la firma de Persona 2

Al verificar la firma de Persona 2 con su llave pública, se obtiene este valor:

{datos_p1_verifica_p2["valor_recuperado_desde_firma"]}

Este valor debe compararse contra el valor preparado del mensaje recibido:

{datos_p1_verifica_p2["valor_preparado_mensaje_recibido"]}
'''
)

print("Valores recuperados guardados correctamente.")

Archivo generado: evidencias\intercambio\03_valor_recuperado_firma_persona_1.txt
Archivo generado: evidencias\intercambio\04_valor_recuperado_firma_persona_2.txt
Valores recuperados guardados correctamente.


## 15. Resumen del intercambio

Se guarda un resumen general del intercambio correcto.

Este archivo sirve para mostrar que ambas personas pudieron verificar el mensaje de la otra usando solo la llave pública correspondiente, el mensaje y la firma.

In [15]:
guardar_txt(
    CARPETA_INTERCAMBIO / "05_resumen_intercambio.txt",
    f'''Resumen del intercambio

Persona 1 firma su propio mensaje con su llave privada.
Persona 2 firma su propio mensaje con su llave privada.

Persona 1 verifica el mensaje de Persona 2 usando la llave pública de Persona 2:
{resultado_p1_verifica_p2} - {explicacion_p1_verifica_p2}

Persona 2 verifica el mensaje de Persona 1 usando la llave pública de Persona 1:
{resultado_p2_verifica_p1} - {explicacion_p2_verifica_p1}

La llave privada no se comparte.
Lo que se comparte para verificar es el mensaje, la firma y la llave pública.
'''
)

print("Resumen del intercambio generado.")

Archivo generado: evidencias\intercambio\05_resumen_intercambio.txt
Resumen del intercambio generado.


## 16. Prueba inválida: mensaje de Persona 1 alterado

Aquí se usa la firma original de Persona 1, pero el mensaje se modifica.

El resultado esperado es que la verificación falle.

In [16]:
mensaje_1_alterado = "Hola Persona 2, este mensaje fue alterado después de ser firmado por Persona 1."

resultado_mensaje_1_alterado, explicacion_mensaje_1_alterado, datos_mensaje_1_alterado = verificar_firma(
    mensaje_1_alterado,
    firma_1,
    llave_publica_1
)

guardar_txt(CARPETA_PRUEBAS_INVALIDAS / "01_mensaje_persona_1_alterado.txt", f"Mensaje alterado de Persona 1:\n\n{mensaje_1_alterado}\n")

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "02_verificacion_mensaje_persona_1_alterado.txt",
    formato_verificacion(
        "Verificación del mensaje alterado de Persona 1",
        mensaje_1_alterado,
        firma_1,
        llave_publica_1,
        resultado_mensaje_1_alterado,
        explicacion_mensaje_1_alterado,
        datos_mensaje_1_alterado
    )
)

print(resultado_mensaje_1_alterado, explicacion_mensaje_1_alterado)

Archivo generado: evidencias\pruebas_invalidas\01_mensaje_persona_1_alterado.txt
Archivo generado: evidencias\pruebas_invalidas\02_verificacion_mensaje_persona_1_alterado.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 17. Prueba inválida: mensaje de Persona 2 alterado

Aquí se usa la firma original de Persona 2, pero el mensaje se modifica.

El resultado esperado es que la verificación falle.

In [17]:
mensaje_2_alterado = "Hola Persona 1, este mensaje fue alterado después de ser firmado por Persona 2."

resultado_mensaje_2_alterado, explicacion_mensaje_2_alterado, datos_mensaje_2_alterado = verificar_firma(
    mensaje_2_alterado,
    firma_2,
    llave_publica_2
)

guardar_txt(CARPETA_PRUEBAS_INVALIDAS / "03_mensaje_persona_2_alterado.txt", f"Mensaje alterado de Persona 2:\n\n{mensaje_2_alterado}\n")

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "04_verificacion_mensaje_persona_2_alterado.txt",
    formato_verificacion(
        "Verificación del mensaje alterado de Persona 2",
        mensaje_2_alterado,
        firma_2,
        llave_publica_2,
        resultado_mensaje_2_alterado,
        explicacion_mensaje_2_alterado,
        datos_mensaje_2_alterado
    )
)

print(resultado_mensaje_2_alterado, explicacion_mensaje_2_alterado)

Archivo generado: evidencias\pruebas_invalidas\03_mensaje_persona_2_alterado.txt
Archivo generado: evidencias\pruebas_invalidas\04_verificacion_mensaje_persona_2_alterado.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 18. Prueba inválida: llave pública incorrecta

Aquí se intenta verificar la firma de Persona 1 usando la llave pública de Persona 2.

El resultado esperado es que la verificación falle, porque esa llave pública no corresponde a la llave privada que generó la firma.

In [18]:
resultado_llave_incorrecta, explicacion_llave_incorrecta, datos_llave_incorrecta = verificar_firma(
    mensaje_1,
    firma_1,
    llave_publica_2
)

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "05_verificacion_con_llave_publica_incorrecta.txt",
    formato_verificacion(
        "Verificación con llave pública incorrecta",
        mensaje_1,
        firma_1,
        llave_publica_2,
        resultado_llave_incorrecta,
        explicacion_llave_incorrecta,
        datos_llave_incorrecta
    )
)

print(resultado_llave_incorrecta, explicacion_llave_incorrecta)

Archivo generado: evidencias\pruebas_invalidas\05_verificacion_con_llave_publica_incorrecta.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 19. Prueba inválida: firma mal formada

Aquí se intenta verificar una firma que no es un número.

El resultado esperado es que el programa no se rompa y responda con un mensaje de error controlado.

In [19]:
firma_mal_formada = "firma_no_numerica"

resultado_firma_mal_formada, explicacion_firma_mal_formada, datos_firma_mal_formada = verificar_firma(
    mensaje_1,
    firma_mal_formada,
    llave_publica_1
)

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "06_firma_mal_formada.txt",
    f'''Prueba con firma mal formada

Mensaje:
{mensaje_1}

Firma enviada:
{firma_mal_formada}

Resultado:
{resultado_firma_mal_formada}

Explicación:
{explicacion_firma_mal_formada}
'''
)

print(resultado_firma_mal_formada, explicacion_firma_mal_formada)

Archivo generado: evidencias\pruebas_invalidas\06_firma_mal_formada.txt
False La firma debe ser un número entero.


## 20. Prueba inválida: firma que no corresponde al mensaje

Aquí se intenta verificar el mensaje de Persona 1 usando la firma de Persona 2.

El resultado esperado es que falle, porque esa firma fue generada para otro mensaje.

In [20]:
firma_no_corresponde = firma_2

resultado_firma_no_corresponde, explicacion_firma_no_corresponde, datos_firma_no_corresponde = verificar_firma(
    mensaje_1,
    firma_no_corresponde,
    llave_publica_1
)

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "07_firma_que_no_corresponde.txt",
    formato_verificacion(
        "Verificación con firma que no corresponde al mensaje",
        mensaje_1,
        firma_no_corresponde,
        llave_publica_1,
        resultado_firma_no_corresponde,
        explicacion_firma_no_corresponde,
        datos_firma_no_corresponde
    )
)

print(resultado_firma_no_corresponde, explicacion_firma_no_corresponde)

Archivo generado: evidencias\pruebas_invalidas\07_firma_que_no_corresponde.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 21. Resumen final de pruebas

Esta sección muestra un resumen de todos los resultados.

Los casos correctos deben aparecer como `True`. Los casos inválidos deben aparecer como `False`.

In [21]:
pruebas = [
    ("Persona 1 verifica a Persona 2", resultado_p1_verifica_p2, explicacion_p1_verifica_p2),
    ("Persona 2 verifica a Persona 1", resultado_p2_verifica_p1, explicacion_p2_verifica_p1),
    ("Mensaje de Persona 1 alterado", resultado_mensaje_1_alterado, explicacion_mensaje_1_alterado),
    ("Mensaje de Persona 2 alterado", resultado_mensaje_2_alterado, explicacion_mensaje_2_alterado),
    ("Llave pública incorrecta", resultado_llave_incorrecta, explicacion_llave_incorrecta),
    ("Firma mal formada", resultado_firma_mal_formada, explicacion_firma_mal_formada),
    ("Firma que no corresponde", resultado_firma_no_corresponde, explicacion_firma_no_corresponde),
]

for nombre, resultado, explicacion in pruebas:
    print(f"{nombre}: {resultado} - {explicacion}")

Persona 1 verifica a Persona 2: True - La firma es válida.
Persona 2 verifica a Persona 1: True - La firma es válida.
Mensaje de Persona 1 alterado: False - La firma no corresponde al mensaje o a la llave pública usada.
Mensaje de Persona 2 alterado: False - La firma no corresponde al mensaje o a la llave pública usada.
Llave pública incorrecta: False - La firma no corresponde al mensaje o a la llave pública usada.
Firma mal formada: False - La firma debe ser un número entero.
Firma que no corresponde: False - La firma no corresponde al mensaje o a la llave pública usada.


## 22. Conclusión técnica

Esta implementación demuestra el flujo de firma y verificación digital con un esquema asimétrico.

Cada persona firma usando su propia llave privada. La otra persona puede verificar usando la llave pública correspondiente. Si se modifica el mensaje, se usa una llave incorrecta o se altera la firma, la verificación falla.

La implementación es educativa y no debe usarse en producción porque no incluye padding criptográfico seguro ni generación aleatoria criptográficamente segura.